# The Task description
A supervised learning, binary classification task on an image dataset.

# Important Necessary Libs

In [19]:
%pip install --upgrade pip setuptools wheel

%pip install \
numpy==1.26.4 \
scipy==1.11.4 \
matplotlib==3.8.4 \
seaborn==0.13.2 \
pandas==2.2.2 \
scikit-learn==1.5.1 \
opencv-python==4.10.0.84 \
tensorflow==2.16.1 \
keras==3.3.3 \
kagglehub \

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [20]:
import numpy as np
import scipy
import matplotlib
import seaborn as sns
import pandas as pd
import sklearn
import cv2
import kagglehub
import shutil
from pathlib import Path
import json

"""
requirements = {
    "numpy": ("1.24.0", np.__version__),
    "scipy": ("1.14.1", scipy.__version__),
    "matplotlib": ("3.9.2", matplotlib.__version__),
    "seaborn": ("0.12.0", sns.__version__),
    "pandas": ("2.2.3", pd.__version__),
    "scikit-learn": ("1.5.2", sklearn.__version__),
    "opencv-python": ("4.5.4", cv2.__version__),  # use prefix, not full wheel tag
}

for pkg, (expected, actual) in requirements.items():
    assert actual.startswith(expected), f"{pkg}: expected {expected}, got {actual}"

print("All checked versions match.")
"""

'\nrequirements = {\n    "numpy": ("1.24.0", np.__version__),\n    "scipy": ("1.14.1", scipy.__version__),\n    "matplotlib": ("3.9.2", matplotlib.__version__),\n    "seaborn": ("0.12.0", sns.__version__),\n    "pandas": ("2.2.3", pd.__version__),\n    "scikit-learn": ("1.5.2", sklearn.__version__),\n    "opencv-python": ("4.5.4", cv2.__version__),  # use prefix, not full wheel tag\n}\n\nfor pkg, (expected, actual) in requirements.items():\n    assert actual.startswith(expected), f"{pkg}: expected {expected}, got {actual}"\n\nprint("All checked versions match.")\n'

# Data Preprocessing

## Download Dataset Function

In [21]:
def ensure_pneumonia_dataset(
    dataset_name="paultimothymooney/chest-xray-pneumonia",
    target_dir="data/chest_xray"
):
    # Resolve project root (assumes notebook launched from project root)
    project_root = Path.cwd()
    target_path = project_root / target_dir

    # Check if dataset already exists
    if target_path.exists():
        print(f"Dataset already exists at: {target_path}")
        return target_path

    print("Dataset not found. Downloading...")

    # Download dataset
    downloaded_path = Path(kagglehub.dataset_download(dataset_name))

    # Create parent directory
    target_path.parent.mkdir(parents=True, exist_ok=True)

    # Move dataset
    shutil.move(str(downloaded_path), str(target_path))

    print(f"Dataset stored at: {target_path}")
    return target_path

## Preproccess and Cache Data Function

In [22]:
# JSON schema for processed data storage for future use
json_schema = {
    "title": "Preprocessed Xray Dataset Index",
    "type": "object",
    "properties": {
        "samples": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string"},
                    "split": {"type": "string", "enum": ["train", "val", "test"]},
                    "label": {"type": "integer", "enum": [0, 1]},
                    "array_file": {"type": "string"},
                    "image_size": {
                        "type": "array",
                        "items": {"type": "integer"},
                        "minItems": 2,
                        "maxItems": 2
                    }
                },
                "required": ["filename", "split", "label", "array_file", "image_size"],
                "additionalProperties": False
            }
        }
    },
    "required": ["samples"],
    "additionalProperties": False
}

IMG_SIZES = ((224, 224), (256, 256))

label_map = {
    "NORMAL": np.uint8(0),
    "PNEUMONIA": np.uint8(1)
}

def preprocess_and_cache_dataset(
    raw_base_dir="data/chest_xray/chest_xray",
    processed_base_dir="data/processed",
    img_sizes=IMG_SIZES
):
    raw_base = Path(raw_base_dir)
    processed_base = Path(processed_base_dir)
    index_file = processed_base / "dataset_index.json"

    samples = []

    for img_size in img_sizes:
        width, height = img_size
        size_name = f"{width}x{height}"
        arrays_base = processed_base / "arrays" / size_name

        # Create output folders per size and split
        for split in ["train", "val", "test"]:
            (arrays_base / split).mkdir(parents=True, exist_ok=True)

        for split in ["train", "val", "test"]:
            for class_name in ["NORMAL", "PNEUMONIA"]:
                class_dir = raw_base / split / class_name
                label = int(label_map[class_name])

                if not class_dir.exists():
                    print(f"Skipping missing directory: {class_dir}")
                    continue

                for file_path in class_dir.iterdir():
                    if not file_path.is_file():
                        continue

                    img = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)
                    if img is None:
                        print(f"Skipping unreadable file: {file_path}")
                        continue

                    # preprocess for this image size
                    img_resized = cv2.resize(img, img_size)
                    img_resized = img_resized.astype(np.float32) / 255.0
                    img_resized = np.expand_dims(img_resized, axis=-1)  # (H, W, 1)

                    # safer filename to avoid collisions
                    array_filename = f"{split}_{class_name}_{file_path.stem}.npy"
                    array_rel_path = Path("arrays") / size_name / split / array_filename
                    array_abs_path = processed_base / array_rel_path

                    np.save(array_abs_path, img_resized)

                    samples.append({
                        "filename": file_path.name,
                        "split": split,
                        "label": label,
                        "array_file": str(array_rel_path).replace("\\", "/"),
                        "image_size": [height, width]
                    })

    dataset_index = {"samples": samples}

    processed_base.mkdir(parents=True, exist_ok=True)
    with open(index_file, "w", encoding="utf-8") as f:
        json.dump(dataset_index, f, indent=2)

    print(f"Saved index to: {index_file}")
    print(f"Saved {len(samples)} processed samples across {len(img_sizes)} image sizes.")
    return dataset_index

## Load Data into Memory Function

In [23]:
def load_cached_splits(
    processed_base_dir="data/processed",
    img_size=(224, 224),
    return_filenames=False
):
    processed_base = Path(processed_base_dir)
    index_file = processed_base / "dataset_index.json"

    if not index_file.exists():
        raise FileNotFoundError(f"Index file not found: {index_file}")

    with open(index_file, "r", encoding="utf-8") as f:
        dataset_index = json.load(f)

    target_size = list(img_size)

    X_train, y_train, train_files = [], [], []
    X_val, y_val, val_files = [], [], []
    X_test, y_test, test_files = [], [], []

    for sample in dataset_index["samples"]:
        if sample["image_size"] != target_size:
            continue

        array_path = processed_base / sample["array_file"]
        if not array_path.exists():
            print(f"Skipping missing array file: {array_path}")
            continue

        img = np.load(array_path)
        label = np.uint8(sample["label"])
        filename = sample["filename"]
        split = sample["split"]

        if split == "train":
            X_train.append(img)
            y_train.append(label)
            train_files.append(filename)
        elif split == "val":
            X_val.append(img)
            y_val.append(label)
            val_files.append(filename)
        elif split == "test":
            X_test.append(img)
            y_test.append(label)
            test_files.append(filename)

    X_train = np.array(X_train, dtype=np.float32)
    y_train = np.array(y_train, dtype=np.uint8)

    X_val = np.array(X_val, dtype=np.float32)
    y_val = np.array(y_val, dtype=np.uint8)

    X_test = np.array(X_test, dtype=np.float32)
    y_test = np.array(y_test, dtype=np.uint8)

    if return_filenames:
        return (
            X_train, y_train, train_files,
            X_val, y_val, val_files,
            X_test, y_test, test_files
        )

    return X_train, y_train, X_val, y_val, X_test, y_test

# Main Data Functions Run

In [24]:
def prepare_and_load_pneumonia_data(
    dataset_name="paultimothymooney/chest-xray-pneumonia",
    raw_target_dir="data/chest_xray",
    processed_base_dir="data/processed",
    img_size=(224, 224),
    img_sizes=IMG_SIZES,
    return_filenames=False
):
    # 1. Ensure raw dataset exists
    raw_dataset_path = ensure_pneumonia_dataset(
        dataset_name=dataset_name,
        target_dir=raw_target_dir
    )

    raw_base_dir = Path(raw_dataset_path) / "chest_xray"
    processed_base = Path(processed_base_dir)
    index_file = processed_base / "dataset_index.json"

    # 2. Decide whether preprocessing is needed
    needs_preprocessing = True

    if index_file.exists():
        try:
            with open(index_file, "r", encoding="utf-8") as f:
                dataset_index = json.load(f)

            samples = dataset_index.get("samples", [])

            # expected number of raw images
            raw_count = 0
            for split in ["train", "val", "test"]:
                for class_name in ["NORMAL", "PNEUMONIA"]:
                    class_dir = raw_base_dir / split / class_name
                    if class_dir.exists():
                        raw_count += sum(1 for p in class_dir.iterdir() if p.is_file())

            expected_count = raw_count * len(img_sizes)

            # check count
            count_ok = len(samples) == expected_count

            # check requested image size exists
            size_ok = any(sample["image_size"] == [img_size[1], img_size[0]] for sample in samples)

            # check array files exist
            files_ok = True
            for sample in samples:
                array_path = processed_base / sample["array_file"]
                if not array_path.exists():
                    files_ok = False
                    break

            needs_preprocessing = not (count_ok and size_ok and files_ok)

        except Exception as e:
            print(f"Processed cache check failed: {e}")
            needs_preprocessing = True

    # 3. Preprocess if needed
    if needs_preprocessing:
        print("Processing and caching dataset...")
        preprocess_and_cache_dataset(
            raw_base_dir=str(raw_base_dir),
            processed_base_dir=str(processed_base),
            img_sizes=img_sizes
        )
    else:
        print("Processed cache already valid. Skipping preprocessing.")

    # 4. Load cached splits
    return load_cached_splits(
        processed_base_dir=str(processed_base),
        img_size=img_size,
        return_filenames=return_filenames
    )

In [ ]:
# There are two possible 
X_train, y_train, X_val, y_val, X_test, y_test = prepare_and_load_pneumonia_data(
    img_size=(224, 224)
)

Dataset not found. Downloading...


100%|██████████| 2.29G/2.29G [01:31<00:00, 26.9MB/s]

Extracting files...


OSError: [Errno 30] Read-only file system: '/data'

## Sanity Check Data

## Exploratory Data Analysis

In [10]:
from google.colab.patches import cv2_imshow

print(X_train[0])

# Normal images view
print(cv2_imshow(X_train[1]))



# Pneumonia images view



# Distributions

  # In train data

  # In test data

  # In val data

ModuleNotFoundError: No module named 'google.colab'

## Missing Value Treaments

## Outlier Treatments

## Duplicates and Garbage Value Treatment

## Normalisation

## Encoding

# Baseline

# Hyperparameter Optimisation and Alternative Model.

# Results generation